DATA COLLECTION

In [5]:
import requests
import pandas as pd
import time

API_KEY = "API_KEY"

URL = "https://eventregistry.org/api/v1/article/getArticles"

all_articles = []

# Search phrase
SEARCH_TERM = "MacBook Neo"

for page in range(1, 21):
    print(f"Fetching page {page}...")

    payload = {
        "query": {
            "$query": {

                # STRICT phrase match
                "$and": [
                    {
                        "keyword": SEARCH_TERM,
                        "keywordLoc": "title",
                        "keywordSearchMode": "exact"
                    },
                    {
                        "dateStart": "2026-03-11",
                        "dateEnd": "2026-05-15"
                    },
                    {
                        "lang": "eng"
                    }
                ]
            },

            "$filter": {
                "isDuplicate": "skipDuplicates",
                "dataType": ["news"]
            }
        },

        "articlesPage": page,
        "articlesCount": 100,
        "articlesSortBy": "date",
        "articlesSortByAsc": False,
        "resultType": "articles",
        "articleBodyLen": -1,
        "apiKey": API_KEY
    }

    try:
        response = requests.post(URL, json=payload)

        if response.status_code != 200:
            print(f"Error {response.status_code}")
            print(response.text)
            break

        data = response.json()

        articles = data.get("articles", {}).get("results", [])

        if not articles:
            print("No more articles found.")
            break

        # Local relevance filtering
        for a in articles:

            title = (a.get("title") or "").lower()
            body = (a.get("body") or "").lower()

            # Strong relevance check
            if "macbook neo" not in title and "macbook neo" not in body:
                continue

            all_articles.append({
                "Date": a.get("date"),
                "Source": a.get("source", {}).get("title"),
                "Title": a.get("title"),
                "Sentiment": a.get("sentiment"),
                "Body": a.get("body"),
                "URL": a.get("url")
            })

        time.sleep(1)

    except Exception as e:
        print("Request failed:", e)
        break

# Convert to DataFrame
df = pd.DataFrame(all_articles)

if not df.empty:

    # Format date
    df["Date"] = pd.to_datetime(df["Date"])

    # Remove duplicates
    df = df.drop_duplicates(
        subset=["Title"],
        keep="first"
    )

    # Sort newest first
    df = df.sort_values(
        by="Date",
        ascending=False
    )

    print(f"\nTotal clean articles: {len(df)}")

    # Save CSV
    df.to_csv("macbook_neo_articles.csv", index=False)

    print("CSV saved successfully.")

else:
    print("No relevant articles found.")

Fetching page 1...
Fetching page 2...
Fetching page 3...
Fetching page 4...
No more articles found.

Total clean articles: 276
CSV saved successfully.


In [2]:
print(df)

          Date                   Source  \
0   2026-05-15                  Gizmodo   
2   2026-05-15              Phone Arena   
3   2026-05-15                 Macworld   
4   2026-05-15  shopping.ndtvprofit.com   
5   2026-05-15          VietNamNet News   
..         ...                      ...   
262 2026-04-16             Michael Tsai   
261 2026-04-16             MacDailyNews   
260 2026-04-16                 Wccftech   
259 2026-04-16                MacRumors   
281 2026-04-16          Manila Bulletin   

                                                 Title  Sentiment  \
0    The AI Chip Rush Is Making Hardware More Expen...   0.145098   
2    Did Apple's MacBook Neo accidentally build the...   0.129412   
3    Microsoft commissioned a very serious study to...   0.223529   
4    Apple MacBook Neo 13 Inch A18 Pro Review: Feat...   0.537255   
5    Google reveals AI-first Googlebook laptops to ...   0.247059   
..                                                 ...        ...   
